In [1]:
import cv2
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import json
from mmcv.transforms import Compose
import numpy as np
from mmdet.utils import get_test_pipeline_cfg

def read_json(json_path):
    with open(json_path) as f:
        data = json.load(f)
    return data

def preprocess(test_pipeline, image):
    if isinstance(image, np.ndarray):
        # Calling this method across libraries will result
        # in module unregistered error if not prefixed with mmdet.
        test_pipeline[0].type = 'mmdet.LoadImageFromNDArray'
    test_pipeline = Compose(test_pipeline)
    return test_pipeline(dict(img=image))

class CustomImageDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, annotations_json_path, transform=None):
        self.transform = transform
        self.images_dir = images_dir
        self.annotations_json = read_json(annotations_json_path)


    def __len__(self):
        return len(self.annotations_json['images'])

    def __getitem__(self, idx):
        image_dict = self.annotations_json['images'][idx]
        image_path = os.path.join(self.images_dir, image_dict['file_name'])
        image_id = image_dict['id']

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            transformed_images = self.transform(image)
        else:
            transformed_images = image

        return image_id, image_path, transformed_images


# calibrationDataloader = DataLoader(calibrationDataset, batch_size=32, shuffle=True)

In [2]:
# %cd /content/drive/MyDrive/Aimet-torch/mmdetection-3.3.0/
import torch
from mmdet.apis import DetInferencer

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize([640, 640]),  # Resize
])

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CONFIG_PATH = 'rtmdet_tiny_8xb32-300e_coco.py'
WEIGHTS_PATH = '/teamspace/studios/this_studio/mmdetection/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth'

ROOT_DATASET_DIR = '/teamspace/studios/aimet/COCO'
IMAGES_DIR = os.path.join(ROOT_DATASET_DIR, 'images')
ANNOTATIONS_JSON_PATH = os.path.join(ROOT_DATASET_DIR, 'annotations/instances_val2017.json')
# ANNOTATIONS_JSON_PATH = "/home/shayaan/Desktop/aimet/my_mmdet/temp.json"


model = DetInferencer(model=CONFIG_PATH, weights=WEIGHTS_PATH, device=DEVICE)
evalDataset = CustomImageDataset(images_dir=IMAGES_DIR, annotations_json_path=ANNOTATIONS_JSON_PATH, transform=transform)
DEVICE

[2024-08-20 10:21:06,025] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  NVIDIA Inference is only supported on Ampere and newer architectures
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.2
 [WARNING]  using untested triton version (2.2.0), only 1.0.0 is known to be compatible
2024-08-20 10:21:08,523 - root - INFO - AIMET
Loads checkpoint by local backend from path: /teamspace/studios/this_studio/mmdetection/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: data_preprocessor.mean, data_preprocessor.std

08/20 10:21:23 

/teamspace/studios/this_studio/mmengine/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '


device(type='cuda', index=0)

In [3]:
bbox_head = model.model.bbox_head
type(bbox_head)

mmdet.models.dense_heads.rtmdet_head.RTMDetSepBNHead

In [4]:
from mmcv.transforms import Compose
test_evaluator = model.cfg.test_evaluator
test_evaluator.type = 'mmdet.evaluation.CocoMetric' 
test_evaluator.dataset_meta = model.model.dataset_meta
test_evaluator.ann_file = ANNOTATIONS_JSON_PATH
test_evaluator = Compose(test_evaluator)

loading annotations into memory...
Done (t=0.84s)
creating index...
index created!


In [5]:
import random
from typing import Optional
from tqdm.notebook import tqdm
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from mmengine.structures import InstanceData

EVAL_DATASET_SIZE = 5000
CALIBRATION_DATASET_SIZE = 2000
BATCH_SIZE = 64

In [6]:
from mmdet.models.utils import samplelist_boxtype2tensor
from mmengine.registry import MODELS

collate_preprocessor = model.preprocess
predict_by_feat = model.model.bbox_head.predict_by_feat
rescale = True

preprocessor = MODELS.build(model.cfg.model.data_preprocessor)
def add_pred_to_datasample(data_samples, results_list):
    for data_sample, pred_instances in zip(data_samples, results_list):
        data_sample.pred_instances = pred_instances
    samplelist_boxtype2tensor(data_samples)
    return data_samples

In [8]:
from aimet_torch.quantsim import QuantizationSimModel, QuantScheme
import torch
from aimet_torch.model_preparer import prepare_model

dummy_input = torch.rand(1, 3, 640, 640).to(DEVICE)
# BASE_PATH = "/teamspace/studios/this_studio/mmdetection/rtm_weights/gpu19aug/rtm_det_gpu_16Aug_rangelearning_20240819_155611_300"
# BASE_PATH = "/teamspace/studios/this_studio/mmdetection/rtm_weights/gpu19aug/rtm_det_gpu_16Aug_rangelearning_20240819_155611_300"
# BASE_PATH = "/teamspace/studios/this_studio/mmdetection/rtm_weights/gpu20aug_neck/rtm_det_gpu_20aug_20240820_073659_300"
BASE_PATH = "/teamspace/studios/this_studio/mmdetection/rtm_weights/gpu20aug_neck_range/rtm_det_gpu_20aug_20240820_100350_350"

graph_model = torch.load(f"{BASE_PATH}/rtm_det.pth", map_location=DEVICE)

quant_scheme = QuantScheme.training_range_learning_with_tf_enhanced_init
# quant_scheme = QuantScheme.post_training_tf_enhanced
sim = QuantizationSimModel(model=graph_model,
                        quant_scheme=quant_scheme,
                        dummy_input=dummy_input,
                        default_output_bw=8,
                        default_param_bw=8,)
sim.load_encodings(f"{BASE_PATH}/rtm_det_torch.encodings")

2024-08-20 10:21:49,141 - Quant - INFO - No config file provided, defaulting to config file at /usr/local/lib/python3.10/dist-packages/aimet_common/quantsim_config/default_config.json
2024-08-20 10:21:49,173 - Quant - INFO - Unsupported op type Squeeze
2024-08-20 10:21:49,173 - Quant - INFO - Unsupported op type Mean
2024-08-20 10:21:49,185 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:default


In [9]:
# from aimet_torch.quantsim import load_checkpoint
# import torch
# sim = load_checkpoint("/teamspace/studios/this_studio/mmdetection/rtm_weights/gpu19aug_range_lr/rtm_det_gpu_16Aug_rangelearning_20240819_105524_300")


In [9]:
# dummy_input = torch.rand(1, 3, 640, 640)
# import os
# os.makedirs("./temp", exist_ok=True)
# sim.export(path="./temp",
#     filename_prefix="rtm_det",
#     dummy_input=dummy_input.cpu(),
#     use_embedded_encodings=True)

In [10]:
from aimet_torch.auto_quant import AutoQuant
from glob import glob

def new_eval_callback(model: torch.nn.Module, num_samples: Optional[int] = None) -> float:
    data_loader = DataLoader(evalDataset, batch_size=BATCH_SIZE)
    new_preds = []
    for image_id, image_path, _ in tqdm(data_loader):
        pre_processed = collate_preprocessor(inputs=image_path, batch_size=BATCH_SIZE)
        _, data = list(pre_processed)[0]
        data = preprocessor(data, False)
        with torch.no_grad():
            preds = model(data['inputs'].to(DEVICE))
            preds = bbox_head(preds)
            
            batch_img_metas = [
            data_samples.metainfo for data_samples in data['data_samples']
            ]
            preds = predict_by_feat(*preds, batch_img_metas=batch_img_metas, rescale=True)
            preds = add_pred_to_datasample(data['data_samples'], preds)
            
            for img_id, pred in zip(image_id, preds):
                pred = pred.pred_instances
                new_pred = InstanceData(metainfo={"img_id": int(img_id)})
                new_pred.bboxes = [np.array(p) for p in pred['bboxes'].to('cpu')]
                new_pred.labels = pred['labels'].to('cpu')
                new_pred.scores = pred['scores'].to('cpu')
                new_preds.append(new_pred)

    eval_results = test_evaluator(new_preds)
    num_file = len(glob(f"{BASE_PATH}/eval_acc_*"))
    with open(f"{BASE_PATH}/eval_acc_{num_file}.json", "w") as f:
        json.dump(eval_results, f, indent=4)
    bbox_map = eval_results['bbox_mAP']
    return bbox_map


In [11]:
sim_model = sim.model.eval()
# print(model)
new_eval_callback(sim_model)


  0%|          | 0/79 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/20 10:37:48 - mmengine - INFO - Evaluating bbox...
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=10.42s).
Accumulating evaluation results...
DONE (t=2.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Reca

0.0